# Vaksinasi dan Beban Layanan Kesehatan

**ID proyek:** `O005-LEGA-V101-PRJ04`  
**Status:** titik awal pedagogis yang ditulis secara independen.

Notebook ini menggunakan data sintetis/terbuka saja. Notebook ini **bukan** kode atau data dari makalah yang dikutip dalam bab sumber dan **bukan** klaim reproduksi hasil penelitian mana pun.


## Pertanyaan pemodelan

Bagaimana cakupan vaksin mengubah puncak kebutuhan perawatan dalam model transparan yang memisahkan perlindungan terhadap infeksi dan penyakit berat?

Tujuan kerja: tetapkan sistem, jalankan eksperimen deterministik, periksa invarian, visualisasikan perilaku, lalu kritik kecukupan model.


In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

SEED = 2026082204
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=6, suppress=True)


## Struktur dan asumsi

Kekebalan awal proporsional terhadap cakupan dan efektivitas terhadap infeksi; risiko perawatan pada kasus terobosan berkurang secara linier; dinamika berikutnya mengikuti SIR.

Semua skala dan parameter di notebook ini bersifat ilustratif. Ubah satu asumsi pada satu waktu dan catat dampaknya pada keluaran serta invarian.


In [ ]:
beta, gamma = 0.36, 0.11
ve_infection, ve_severe, base_hospitalization = 0.70, 0.85, 0.08
t_eval = np.linspace(0.0, 160.0, 641)

def vaccinated_run(coverage):
    immune = coverage * ve_infection
    I0 = 0.001
    y0 = [1.0 - immune - I0, I0, immune]
    def rhs(t, y):
        S, I, R = y
        return [-beta * S * I, beta * S * I - gamma * I, gamma * I]
    run = solve_ivp(rhs, (0.0, 160.0), y0, t_eval=t_eval, rtol=1e-9, atol=1e-11)
    severe_multiplier = 1.0 - coverage * ve_severe
    burden = base_hospitalization * severe_multiplier * run.y[1]
    return run, burden

coverage_runs = {coverage: vaccinated_run(coverage) for coverage in (0.0, 0.5, 0.8)}
burden_peaks = {coverage: float(burden.max()) for coverage, (_, burden) in coverage_runs.items()}


## Pemeriksaan numerik

Pemeriksaan berikut sengaja berada di dalam notebook: eksekusi berhenti bila suatu invarian dasar gagal. Ini bukan bukti bahwa model benar; ini hanya bukti bahwa implementasi memenuhi kontrak numerik terbatasnya.


In [ ]:
for run, burden in coverage_runs.values():
    assert run.success and np.min(run.y) > -1e-9 and np.min(burden) >= 0.0
    np.testing.assert_allclose(run.y.sum(axis=0), 1.0, atol=2e-8)
assert burden_peaks[0.8] < burden_peaks[0.5] < burden_peaks[0.0]


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
for coverage, (_, burden) in coverage_runs.items():
    ax.plot(t_eval, 100000 * burden, label=f"cakupan {coverage:.0%}")
ax.axhline(200, color="black", linestyle="--", linewidth=1, label="kapasitas ilustratif")
ax.set(xlabel="hari", ylabel="kebutuhan perawatan per 100.000", title="Beban layanan pada beberapa cakupan")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()
plt.close(fig)


## Validasi, identifikasi, dan keterbatasan

Keterbatasan awal: Tidak ada peluruhan kekebalan, kelompok umur, kapasitas yang memengaruhi mortalitas, dosis berulang, atau seleksi varian.

Jawab sebelum menafsirkan gambar:

1. Besaran apa yang benar-benar dapat diamati, dan bagaimana galat pengukurannya dimodelkan?
2. Parameter mana yang dapat diidentifikasi dari keluaran tersebut? Tunjukkan dengan profil galat, pemisahan latih/uji, atau eksperimen sensitivitas.
3. Invarian atau pola kualitatif apa yang harus tetap benar ketika ukuran langkah, benih acak, atau resolusi diubah?
4. Temukan satu skenario kegagalan model dan jelaskan data tambahan yang diperlukan untuk membedakannya dari model alternatif.


## Daftar periksa reproduksibilitas

- [ ] Gunakan CPython dan versi paket tepat seperti `requirements.lock`.
- [ ] Jalankan ulang dari kernel kosong tanpa jaringan.
- [ ] Pertahankan nilai `SEED` (benih acak), lalu ulangi dengan sedikitnya lima benih acak lain dan laporkan variasinya.
- [ ] Catat setiap perubahan parameter, persamaan, toleransi, serta pembagian data.
- [ ] Pastikan semua uji lulus dan jelaskan mengapa tiap uji relevan.
- [ ] Simpan hasil turunan di luar notebook sumber; notebook distribusi harus tetap tanpa keluaran tersimpan.
- [ ] Bedakan hasil simulasi, data sintetis, dan klaim empiris secara eksplisit.
